<a href="https://colab.research.google.com/github/rickyp1971/chatbot-AI-asisten-Cuaca/blob/main/final_chatbot_weather_terbaru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U streamlit pyngrok requests langchain-core langchain-community langchain-google-genai faiss-cpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take 

In [ ]:
# Dependensi sudah dipasang pada sel pertama.


In [ ]:
# Dependensi requests sudah dipasang pada sel pertama.


In [5]:
import os
from google.colab import userdata
from pyngrok import ngrok

def get_colab_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
            if value and str(value).strip():
                return str(value).strip()
        except Exception:
            pass
    return ""

# Buat secret berikut di Google Colab > Secrets (ikon kunci).
# Nama yang didukung:
# Gemini: GEMINI_API_KEY atau GOOGLE_API_KEY
# Cuaca: OPENWEATHER_API_KEY
# Ngrok: NGROK_AUTHTOKEN atau ngrok
gemini_key = get_colab_secret("gemini", "GOOGLE_API_KEY")
weather_key = get_colab_secret("weather")
ngrok_token = get_colab_secret("NGROK_AUTHTOKEN", "ngrok")

if not gemini_key:
    raise RuntimeError("Secret GEMINI_API_KEY atau GOOGLE_API_KEY belum tersedia di Colab Secrets.")
if not weather_key:
    raise RuntimeError("Secret OPENWEATHER_API_KEY belum tersedia di Colab Secrets.")
if not ngrok_token:
    raise RuntimeError("Secret NGROK_AUTHTOKEN atau ngrok belum tersedia di Colab Secrets.")

# Environment variables diwariskan oleh proses Streamlit yang diluncurkan notebook.
os.environ["GOOGLE_API_KEY"] = gemini_key
os.environ["OPENWEATHER_API_KEY"] = weather_key
os.environ["NGROK_AUTHTOKEN"] = ngrok_token

ngrok.set_auth_token(ngrok_token)
print("✅ Secret Gemini, OpenWeatherMap, dan ngrok berhasil dimuat. Nilai key tidak ditampilkan.")


✅ Secret Gemini, OpenWeatherMap, dan ngrok berhasil dimuat. Nilai key tidak ditampilkan.


In [6]:
import os
import sys
import time
import subprocess
import urllib.request
from pyngrok import ngrok, conf

def run_streamlit(filename="streamlit_app_basic.py", port=8501):
    if not os.path.isfile(filename):
        raise FileNotFoundError(f"File tidak ditemukan: {filename}")

    token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
    if not token:
        raise RuntimeError("Jalankan sel konfigurasi Colab Secrets terlebih dahulu.")

    # Tutup tunnel pyngrok yang tersisa; tidak membunuh semua proses notebook.
    try:
        for tunnel_info in ngrok.get_tunnels():
            ngrok.disconnect(tunnel_info.public_url)
    except Exception:
        pass

    config = conf.PyngrokConfig(auth_token=token, startup_timeout=30)
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", filename,
            "--server.port", str(port),
            "--server.address", "127.0.0.1",
            "--server.headless", "true",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    ready = False
    for _ in range(40):
        if proc.poll() is not None:
            output = proc.stdout.read() if proc.stdout else ""
            raise RuntimeError("Streamlit berhenti saat startup:\n" + output[-5000:])
        try:
            with urllib.request.urlopen(
                f"http://127.0.0.1:{port}/_stcore/health", timeout=2
            ) as response:
                if response.status == 200:
                    ready = True
                    break
        except Exception:
            pass
        time.sleep(1)

    if not ready:
        proc.terminate()
        output = ""
        try:
            output = proc.stdout.read() if proc.stdout else ""
        except Exception:
            pass
        raise RuntimeError("Streamlit belum siap. Log:\n" + output[-5000:])

    try:
        tunnel = ngrok.connect(
            addr=port,
            proto="http",
            pyngrok_config=config,
        )
        print("✅ Streamlit aktif:", tunnel.public_url)
        return proc
    except Exception:
        proc.terminate()
        raise


In [ ]:
# Launcher run_streamlit sudah didefinisikan pada sel sebelumnya. Jangan jalankan launcher duplikat.


In [ ]:
# Tunnel ngrok akan dibuat otomatis oleh run_streamlit() pada sel terakhir.


In [18]:
# Dependensi sudah dipasang pada sel pertama.
from pyngrok import ngrok

try:
    # Hentikan tunnel lama jika ada
    ngrok.kill()

    # Buat tunnel ke port Streamlit
    tunnel = ngrok.connect(8501, "http")

    print("Aplikasi Streamlit dapat dibuka di:")
    print(tunnel.public_url)

except Exception as e:
    print("Ngrok gagal dijalankan:")
    print(e)

Aplikasi Streamlit dapat dibuka di:
https://dyslexic-scrambled-synthesis.ngrok-free.dev


In [12]:
%%writefile streamlit_app_basic.py
import os
import requests
import streamlit as st
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

st.set_page_config(page_title="Asisten Cuaca AI", page_icon="⛅", layout="wide")

COUNTRIES = {'Indonesia': ('ID', ['Jakarta', 'Bandung', 'Surabaya', 'Yogyakarta', 'Medan', 'Denpasar', 'Makassar', 'Semarang', 'Palembang', 'Balikpapan', 'Malang', 'Bogor']), 'Malaysia': ('MY', ['Kuala Lumpur', 'George Town', 'Johor Bahru', 'Kota Kinabalu', 'Kuching', 'Malacca']), 'Singapura': ('SG', ['Singapore']), 'Thailand': ('TH', ['Bangkok', 'Chiang Mai', 'Phuket', 'Pattaya', 'Hat Yai']), 'Filipina': ('PH', ['Manila', 'Quezon City', 'Cebu City', 'Davao City', 'Iloilo City']), 'Vietnam': ('VN', ['Hanoi', 'Ho Chi Minh City', 'Da Nang', 'Hue', 'Can Tho']), 'Jepang': ('JP', ['Tokyo', 'Osaka', 'Kyoto', 'Yokohama', 'Sapporo', 'Fukuoka', 'Nagoya']), 'Korea Selatan': ('KR', ['Seoul', 'Busan', 'Incheon', 'Daegu', 'Daejeon']), 'Tiongkok': ('CN', ['Beijing', 'Shanghai', 'Guangzhou', 'Shenzhen', 'Chengdu', 'Wuhan']), 'India': ('IN', ['New Delhi', 'Mumbai', 'Bengaluru', 'Chennai', 'Kolkata', 'Hyderabad']), 'Australia': ('AU', ['Sydney', 'Melbourne', 'Brisbane', 'Perth', 'Adelaide', 'Canberra']), 'Selandia Baru': ('NZ', ['Auckland', 'Wellington', 'Christchurch', 'Hamilton']), 'Amerika Serikat': ('US', ['New York', 'Los Angeles', 'Chicago', 'Houston', 'San Francisco', 'Seattle', 'Miami', 'Washington']), 'Kanada': ('CA', ['Toronto', 'Vancouver', 'Montreal', 'Ottawa', 'Calgary']), 'Meksiko': ('MX', ['Mexico City', 'Guadalajara', 'Monterrey', 'Cancun']), 'Brasil': ('BR', ['Sao Paulo', 'Rio de Janeiro', 'Brasilia', 'Salvador', 'Manaus']), 'Argentina': ('AR', ['Buenos Aires', 'Cordoba', 'Rosario', 'Mendoza']), 'Inggris': ('GB', ['London', 'Manchester', 'Birmingham', 'Liverpool', 'Edinburgh', 'Bristol']), 'Prancis': ('FR', ['Paris', 'Marseille', 'Lyon', 'Toulouse', 'Nice']), 'Jerman': ('DE', ['Berlin', 'Munich', 'Hamburg', 'Frankfurt', 'Cologne']), 'Italia': ('IT', ['Rome', 'Milan', 'Naples', 'Florence', 'Venice']), 'Spanyol': ('ES', ['Madrid', 'Barcelona', 'Valencia', 'Seville', 'Malaga']), 'Belanda': ('NL', ['Amsterdam', 'Rotterdam', 'The Hague', 'Utrecht', 'Eindhoven']), 'Turki': ('TR', ['Istanbul', 'Ankara', 'Izmir', 'Antalya']), 'Arab Saudi': ('SA', ['Riyadh', 'Jeddah', 'Mecca', 'Medina', 'Dammam']), 'Uni Emirat Arab': ('AE', ['Dubai', 'Abu Dhabi', 'Sharjah', 'Al Ain']), 'Mesir': ('EG', ['Cairo', 'Alexandria', 'Giza', 'Luxor']), 'Afrika Selatan': ('ZA', ['Johannesburg', 'Cape Town', 'Durban', 'Pretoria']), 'Nigeria': ('NG', ['Lagos', 'Abuja', 'Kano', 'Ibadan']), 'Kenya': ('KE', ['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru'])}

st.markdown("""
<style>
.block-container {max-width:1100px;padding-top:1.4rem;padding-bottom:3rem}
.hero {background:linear-gradient(125deg,#563d7c,#007bff);padding:28px;border-radius:18px;color:white;margin-bottom:22px}
.hero h1,.hero p {color:white!important}
.hero p {font-size:1.02rem;margin-bottom:0}
div[data-testid="stMetric"] {background:var(--secondary-background-color);padding:14px;border-radius:14px;border:1px solid rgba(128,128,128,.18)}
div[data-testid="stChatMessage"] {border-radius:14px}
</style>
<div class="hero">
<h1>⛅ Asisten Cuaca AI</h1>
<p>Cuaca terkini, analisis cerdas, rekomendasi pakaian, dan saran aktivitas.</p>
</div>
""", unsafe_allow_html=True)

# API key dibaca dari environment yang diinjeksi notebook Google Colab.
# Jangan menyimpan API key langsung di file ini.
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY", "").strip()
WEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY", "").strip()
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.6-flash").strip()

with st.sidebar:
    st.header("⚙️ Pengaturan")
    st.caption("API key dimuat otomatis dari Google Colab Secrets.")
    st.caption("Gemini: " + ("✅ terdeteksi" if GEMINI_API_KEY else "❌ secret belum tersedia"))
    st.caption("OpenWeatherMap: " + ("✅ terdeteksi" if WEATHER_API_KEY else "❌ secret belum tersedia"))
    st.markdown("---")
    st.subheader("🌏 Lokasi")
    selected_country = st.selectbox("Pilih negara", list(COUNTRIES.keys()), index=0)
    country_code, city_options = COUNTRIES[selected_country]
    selected_city = st.selectbox("Pilih kota", city_options, index=0, key=f"city_{country_code}")
    st.caption(f"Negara: {selected_country} ({country_code})")
    st.markdown("---")
    if st.button("🗑️ Hapus riwayat chat", use_container_width=True):
        st.session_state.messages = []
        st.rerun()
    st.caption(f"Model AI: {MODEL_NAME}")

WEATHER_DOCS = [
    Document(page_content="Indonesia: iklim tropis, cuaca dan kelembapan berbeda antarwilayah.", metadata={"country":"ID"}),
    Document(page_content="Malaysia dan Singapura: iklim tropis, panas lembap dan hujan lokal mungkin terjadi.", metadata={"country":"MY"}),
    Document(page_content="Thailand, Filipina, dan Vietnam: iklim tropis; kondisi aktual mengikuti data cuaca saat ini.", metadata={"country":"TH"}),
    Document(page_content="Jepang dan Korea Selatan: kondisi cuaca sangat dipengaruhi musim dan lokasi.", metadata={"country":"JP"}),
    Document(page_content="Australia dan Selandia Baru: kondisi cuaca bervariasi antarwilayah dan musim.", metadata={"country":"AU"}),
    Document(page_content="Eropa dan Amerika Utara: cuaca bervariasi berdasarkan wilayah, musim, dan kondisi atmosfer.", metadata={"country":"US"}),
]

@st.cache_resource(show_spinner=False)
def init_vector_db(api_key):
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=api_key,
    )
    return FAISS.from_documents(WEATHER_DOCS, embeddings)

def get_realtime_weather(city, country_code, api_key):
    if not api_key:
        return {"ok": False, "message": "Secret OPENWEATHER_API_KEY belum tersedia."}
    try:
        response = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={"q": f"{city},{country_code}", "appid": api_key, "units": "metric", "lang": "id"},
            timeout=15,
        )
        try:
            data = response.json()
        except ValueError:
            data = {}
        if response.status_code != 200:
            message = data.get("message", f"HTTP {response.status_code}")
            return {"ok": False, "message": f"Data cuaca gagal diambil: {message}"}
        main = data.get("main", {})
        weather = (data.get("weather") or [{}])[0]
        wind = data.get("wind", {})
        return {
            "ok": True,
            "city": data.get("name", city),
            "country": data.get("sys", {}).get("country", country_code),
            "temp": main.get("temp"),
            "feels_like": main.get("feels_like"),
            "humidity": main.get("humidity"),
            "pressure": main.get("pressure"),
            "description": weather.get("description", "Tidak diketahui").capitalize(),
            "wind_speed": wind.get("speed"),
            "visibility": data.get("visibility"),
            "icon": weather.get("icon"),
        }
    except requests.Timeout:
        return {"ok": False, "message": "Koneksi cuaca timeout. Silakan coba lagi."}
    except requests.RequestException:
        return {"ok": False, "message": "Tidak dapat terhubung ke layanan cuaca. Periksa internet."}

def weather_to_text(weather, city, country):
    if not weather.get("ok"):
        return f"Data cuaca aktual untuk {city}, {country} tidak tersedia. {weather.get('message')}. Jangan mengarang data cuaca."
    return (
        f"Lokasi: {weather['city']}, {weather['country']}\n"
        f"Suhu: {weather['temp']}°C\nTerasa seperti: {weather['feels_like']}°C\n"
        f"Kondisi: {weather['description']}\nKelembapan: {weather['humidity']}%\n"
        f"Tekanan: {weather['pressure']} hPa\nAngin: {weather['wind_speed']} m/s\n"
        f"Jarak pandang: {weather['visibility']} meter"
    )

def render_weather(weather, city):
    st.subheader(f"📍 Cuaca di {city}")
    if not weather.get("ok"):
        st.warning(weather.get("message", "Data cuaca tidak tersedia."))
        return
    if weather.get("icon"):
        st.image(f"https://openweathermap.org/img/wn/{weather['icon']}@2x.png", width=78)
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("🌡️ Suhu", f"{weather['temp']}°C")
    c2.metric("🤗 Terasa", f"{weather['feels_like']}°C")
    c3.metric("💧 Kelembapan", f"{weather['humidity']}%")
    c4.metric("🌬️ Angin", f"{weather['wind_speed']} m/s")
    st.info(f"**Kondisi saat ini:** {weather['description']}")
    st.caption("Sumber data: OpenWeatherMap. Ini kondisi saat ini, bukan prakiraan untuk hari-hari berikutnya.")

def extract_text(content):
    # LangChain biasanya mengembalikan string atau list blok teks.
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and isinstance(item.get("text"), str):
                parts.append(item["text"])
        return "\n\n".join(x.strip() for x in parts if x.strip())
    if isinstance(content, dict):
        if isinstance(content.get("text"), str):
            return content["text"].strip()
        return ""
    return str(content).strip()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

query = st.chat_input(f"Tanyakan cuaca di {selected_city}, {selected_country}...")
if query:
    st.session_state.messages.append({"role": "user", "content": query})
    with st.chat_message("user"):
        st.markdown(query)
    with st.chat_message("assistant"):
        if not GEMINI_API_KEY:
            st.error("Secret Gemini belum tersedia. Tambahkan GEMINI_API_KEY atau GOOGLE_API_KEY di Google Colab Secrets, lalu jalankan ulang sel konfigurasi.")
        else:
            with st.spinner("⛅ Mengambil data cuaca dan menyiapkan jawaban..."):
                weather = get_realtime_weather(selected_city, country_code, WEATHER_API_KEY)
                render_weather(weather, selected_city)
                weather_context = weather_to_text(weather, selected_city, selected_country)
                db_context = "Referensi tambahan tidak tersedia."
                try:
                    vectorstore = init_vector_db(GEMINI_API_KEY)
                    docs = vectorstore.as_retriever(search_kwargs={"k": 2}).invoke(
                        f"{selected_country} {selected_city}: {query}"
                    )
                    if docs:
                        db_context = "\n".join(d.page_content for d in docs)
                except Exception as e:
                    st.caption("Referensi FAISS dilewati; jawaban tetap memakai data cuaca jika tersedia.")
                    print("FAISS detail:", repr(e))
                prompt = ChatPromptTemplate.from_messages([
                    ("system", """Anda adalah Asisten Cuaca AI yang ramah dan informatif. Jawab dalam bahasa Indonesia yang mudah dipahami. Gunakan judul Markdown, paragraf pendek, dan daftar bila berguna. Pakai data aktual sebagai sumber utama; jangan mengarang suhu atau kondisi. Jika data aktual tidak tersedia, jelaskan. Referensi klimatologi hanya konteks umum, bukan data real-time. Jangan menampilkan metadata atau struktur internal respons.

NEGARA: {country}
KOTA: {city}
DATA CUACA AKTUAL:
{weather_context}

REFERENSI UMUM:
{db_context}"""),
                    ("human", "{input}")
                ])
                try:
                    llm = ChatGoogleGenerativeAI(
                        model=MODEL_NAME,
                        temperature=0.3,
                        google_api_key=GEMINI_API_KEY,
                        max_retries=2,
                        timeout=60,
                    )
                    prompt_value = prompt.invoke({
                        "country": selected_country,
                        "city": selected_city,
                        "weather_context": weather_context,
                        "db_context": db_context,
                        "input": query,
                    })
                    response = llm.invoke(prompt_value)
                    reply = extract_text(response.content)
                    if reply:
                        st.markdown(reply)
                        st.session_state.messages.append({"role": "assistant", "content": reply})
                    else:
                        st.warning("AI tidak mengembalikan teks. Silakan coba lagi.")
                except Exception as e:
                    st.error("Jawaban AI gagal dibuat. Periksa model, kuota, dan konfigurasi API.")
                    with st.expander("Detail teknis untuk pengembang"):
                        st.code(repr(e))


Overwriting streamlit_app_basic.py


In [13]:
import os
app_file = "streamlit_app_basic.py"
if os.path.isfile(app_file):
    print(f"✅ File aplikasi ditemukan ({os.path.getsize(app_file):,} byte).")
else:
    print("❌ File aplikasi belum ditemukan. Jalankan sel %%writefile terlebih dahulu.")


✅ File aplikasi ditemukan (13,070 byte).


In [17]:
# Jangan membuat tunnel kedua di sel ini; run_streamlit() membuatnya otomatis.
from pyngrok import ngrok

try:
    # Hentikan tunnel lama jika ada
    ngrok.kill()

    # Buat tunnel ke port Streamlit
    tunnel = ngrok.connect(8501, "http")

    print("Aplikasi Streamlit dapat dibuka di:")
    print(tunnel.public_url)

except Exception as e:
    print("Ngrok gagal dijalankan:")
    print(e)

Aplikasi Streamlit dapat dibuka di:
https://dyslexic-scrambled-synthesis.ngrok-free.dev


In [19]:
# Pastikan sel konfigurasi Secrets, launcher, dan %%writefile sudah dijalankan.
proc = run_streamlit("streamlit_app_basic.py")


✅ Streamlit aktif: https://dyslexic-scrambled-synthesis.ngrok-free.dev
